In [ ]:
import types
import pandas as pd
import ibm_boto3
from botocore.client import Config

import os
import pickle
from getpass import getpass

from ibm_watson_studio_lib import access_project_or_space
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from ibm_watson_machine_learning import APIClient
from ibm_watsonx_ai import APIClient, Credentials


def __iter__(self): return 0

ibm_api_key_id=
ibm_service_instance_id=
ibm_auth_endpoint=
config=
endpoint_url=

# Read excel file from preproduction connection
cicd_cos_pre_client = ibm_boto3.client(
    service_name='s3',
    ibm_api_key_id=ibm_api_key_id,
    ibm_service_instance_id=ibm_service_instance_id,
    ibm_auth_endpoint=ibm_service_instance_id,
    config=Config(signature_version='oauth'),
    endpoint_url=endpoint_url
)
bucket = 'cicd-bucket-test'
object_key = 'pre.xlsx'

body = cicd_cos_pre_client.get_object(Bucket=bucket, Key=object_key)['Body']

df_0 = pd.read_excel(body.read())
df_0.head(10)

In [ ]:
X = df_0.drop(columns=["target"])
y = df_0["target"]

# Train a decision tree classifier model
model = LogisticRegression(max_iter=1000)
model.fit(X, y)
print(f"Training accuracy: {model.score(X, y):.2f}")

In [ ]:
CPD_VERSION = "5.4"

credentials = Credentials(
    url=os.environ["RUNTIME_ENV_APSX_URL"],
    token=os.environ["USER_ACCESS_TOKEN"],
    instance_id="openshift",
    version=CPD_VERSION,
)
client = APIClient(credentials)

project_id = os.environ["PROJECT_ID"]
client.set.default_project(project_id)


wslib = access_project_or_space()

# Associate with the current project
project_id = os.environ["PROJECT_ID"]
client.set.default_project(project_id)

In [ ]:
# Register the trained model as a Model asset
# client.software_specifications.list()
SOFTWARE_SPEC_NAME = "runtime-25.1-py3.12"

software_spec_id = client.software_specifications.get_id_by_name(SOFTWARE_SPEC_NAME)
print(software_spec_id)
MODEL_TYPE = "scikit-learn_1.6"

metadata = {
    client.repository.ModelMetaNames.NAME: "Logistic Regression Model",
    client.repository.ModelMetaNames.TYPE: MODEL_TYPE,
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: software_spec_id,
}


model_details = client.repository.store_model(model=model, meta_props=metadata)

# Get the registered model ID
model_id = client.repository.get_model_id(model_details)

print("Model asset created:", model_id)
